# 03 — Data Cleaning

# Data Cleaning — NovaPay Transactions

## Purpose

This document outlines the cleaning rules applied to the raw NovaPay transaction
data to produce an analysis-ready dataset for modeling.

## Files

- **Input:** `data/nova_pay_combined.csv` — raw data, never modified.
- **Output:** `data/cleaned_transactions.csv` — cleaned, analysis-ready dataset.
- **Process:** `notebook/03_cleaning.ipynb` — the cleaning notebook.

## Cleaning Rules Applied

**1. Removed duplicate records.**
200 fully duplicated rows were removed, confirmed at the `transaction_id` level.
Each transaction is now represented once.

**2. Standardized timestamps.**
The `timestamp` column was converted from text to a proper datetime type.
Timezone information was removed so all timestamps are consistent and naive.
Values that could not be parsed were treated as missing.

**3. Removed future-dated transactions.**
Transactions with a timestamp later than the current date were removed, as a
transaction cannot occur in the future. These records are invalid.

**4. Normalized text categories.**
The columns `channel`, `kyc_tier`, `home_country`, `source_currency`, and
`dest_currency` were standardized by trimming whitespace and unifying
capitalization, so each real category is represented by a single consistent value.

**5. Corrected known spelling errors.**
Recognized typos were corrected: `weeb` to `web`, `mobille` to `mobile`,
`standrd` to `standard`, and `enhancd` to `enhanced`.

**6. Corrected data types.**
The `amount_src` column was converted from text to numeric. Values that were
not valid numbers were treated as missing and handled in the step below.

**7. Handled missing values.**
- Rows missing a `timestamp` were dropped, as the timestamp is essential.
- Numeric columns (`amount_usd`, `fee`, `device_trust_score`, `amount_src`)
  were filled with the column median, which is not distorted by extreme values.
- Categorical columns (`ip_address`, `ip_country`, `kyc_tier`) were filled with
  the label "unknown", since a missing category can itself be meaningful.

**8. Preserved label integrity.**
The target variable `is_fraud` was never altered, filled, or removed based on
its value. All cleaning was applied to feature columns only.

## Sense Checks Performed

After cleaning, the dataset was checked for numeric consistency:
negative amounts, fees, and scores; fees exceeding transaction amounts;
1-hour transaction velocity exceeding 24-hour velocity; and score columns
outside their expected ranges. The `is_fraud` label distribution was confirmed
to contain only valid values.

## Result

The cleaned dataset is free of duplicates, has consistent categories and data
types, contains no invalid future-dated records, and has all missing values
handled. It is ready for analysis and modeling.





In [46]:
import pandas as pd
import numpy as np
df = pd.read_csv(r'C:\Users\Opeyemi\OneDrive - UNC Kenan-Flagler Business School\Desktop\Data Science\Fraudulent-Transaction-Detection-for-Digital-Money-Transfer\data\nova_pay_combined.csv')

In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  str    
 1   customer_id                11400 non-null  str    
 2   timestamp                  11371 non-null  str    
 3   home_country               11400 non-null  str    
 4   source_currency            11400 non-null  str    
 5   dest_currency              11400 non-null  str    
 6   channel                    11400 non-null  str    
 7   amount_src                 11400 non-null  str    
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  str    
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  str    
 14  i

In [48]:
before = len(df)

In [49]:
df.duplicated().sum()
df = df.drop_duplicates()
after = len(df)

In [50]:
print('Duplicates removed:', before - after)
print('Rows remaining:', after)

Duplicates removed: 200
Rows remaining: 11200


In [51]:
df.info()

<class 'pandas.DataFrame'>
Index: 11200 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11200 non-null  str    
 1   customer_id                11200 non-null  str    
 2   timestamp                  11171 non-null  str    
 3   home_country               11200 non-null  str    
 4   source_currency            11200 non-null  str    
 5   dest_currency              11200 non-null  str    
 6   channel                    11200 non-null  str    
 7   amount_src                 11200 non-null  str    
 8   amount_usd                 10900 non-null  float64
 9   fee                        10910 non-null  float64
 10  exchange_rate_src_to_dest  11200 non-null  float64
 11  device_id                  11200 non-null  str    
 12  new_device                 11200 non-null  bool   
 13  ip_address                 10900 non-null  str    
 14  ip_cou

In [52]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors='coerce')

df["amount_src"] = pd.to_numeric(df["amount_src"], errors='coerce')
df.info()

<class 'pandas.DataFrame'>
Index: 11200 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             11200 non-null  str                
 1   customer_id                11200 non-null  str                
 2   timestamp                  11140 non-null  datetime64[us, UTC]
 3   home_country               11200 non-null  str                
 4   source_currency            11200 non-null  str                
 5   dest_currency              11200 non-null  str                
 6   channel                    11200 non-null  str                
 7   amount_src                 11196 non-null  float64            
 8   amount_usd                 10900 non-null  float64            
 9   fee                        10910 non-null  float64            
 10  exchange_rate_src_to_dest  11200 non-null  float64            
 11  device_id         

In [53]:
df.isnull().sum()

transaction_id                 0
customer_id                    0
timestamp                     60
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     4
amount_usd                   300
fee                          290
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   300
ip_country                   296
location_mismatch              0
ip_risk_score                  0
kyc_tier                     295
account_age_days               0
device_trust_score           290
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

##### For currency related missing values 
###### select rows where amount_usd is present
###### group by source_currency
###### compute the mean of amount_usd/amount_src for each currency, aim is to get the exchange rate
###### convert the result to dictionary for easy look up

In [54]:
exchange_rates = df[df['amount_usd'].notna()].groupby('source_currency').apply(
    lambda x: (x['amount_usd'] / x['amount_src']).mean()
               ).to_dict()


In [55]:
df['source_currency'].value_counts()


source_currency
USD    7875
GBP    2111
CAD    1214
Name: count, dtype: int64

In [56]:
exchange_rates

{'CAD': 0.7226052713792916, 'GBP': 1.224313741881689, 'USD': 0.983814123482574}

#### To fill cells missing usd value based on calculated exchange rates
###### if ammount_usd is present keep it
###### Otherwise calc it using ammount_src* exchange rate for the src
###### defualts to 1 if the currency is not in exchange rates

In [57]:
df['amount_usd'] = df.apply(
    lambda row: row['amount_usd'] if pd.notna(row['amount_usd']) else row['amount_src'] * exchange_rates.get(row['source_currency'],1), 
    axis=1
)

#### Missing fee values
###### if channel exists, fill missing fee per channel using the channet's median
###### then fill any reaming missing fee using the overall median

In [58]:
# fee : median (or by channel if present)
if " fee" in df.columns:
    if ' channel' in df.columns:
         df['fee'] = df.groupby('channel')['fee'].transform(lambda s:s.fillna(s.median()))
df['fee'] = df['fee'].fillna(df['fee'].median())


##### The cell missing ip_country
##### if ip_country is missing, it uses the corresponding home_country as a fallback

In [59]:
# ip_country: fallback to home_country
if {'ip_country', 'home_country'}.issubset(df.columns):
    df['ip_country'] = df['ip_country'].fillna(df['home_country'])

#### The cell missing kyc_tier
##### find the most frequent kyc_tier(mode)
##### if  mode is unavailable, defaults to 'standard'
##### fills missing values with this mode/default

In [60]:
# kyc_tier: fill with mode
if 'kyc_tier' in df.columns:
    mode_kyc = df['kyc_tier'].mode().iloc[0] if not df['kyc_tier'].mode().empty else 'standard'
    df['kyc_tier'] = df['kyc_tier'].fillna(mode_kyc)

### The cell missing device_trust_score
##### if  new_device and kyc_tier exist, fill missing score per group using the group median
##### then fill any remaing missing score with the overall median

In [61]:
if 'device_trust_score' in df.columns:
    if{'new_device', 'kyc_tier'}.issubset(df.columns):
        df['device_trust_score'] = df.groupby(['new_device', 'kyc_tier'])['device_trust_score']\
            .transform(lambda s: s.fillna(s.median()))
    df['device_trust_score'] = df['device_trust_score'].fillna(df['device_trust_score'].median())


In [62]:
df.isna().sum()

transaction_id                 0
customer_id                    0
timestamp                     60
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     4
amount_usd                     0
fee                            0
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   300
ip_country                     0
location_mismatch              0
ip_risk_score                  0
kyc_tier                       0
account_age_days               0
device_trust_score             0
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

In [63]:
df.dropna(inplace=True)

In [64]:
df.isna().sum()

transaction_id               0
customer_id                  0
timestamp                    0
home_country                 0
source_currency              0
dest_currency                0
channel                      0
amount_src                   0
amount_usd                   0
fee                          0
exchange_rate_src_to_dest    0
device_id                    0
new_device                   0
ip_address                   0
ip_country                   0
location_mismatch            0
ip_risk_score                0
kyc_tier                     0
account_age_days             0
device_trust_score           0
chargeback_history_count     0
risk_score_internal          0
txn_velocity_1h              0
txn_velocity_24h             0
corridor_risk                0
is_fraud                     0
dtype: int64

##### Sanity check list for Fraud detection (for any inconsistencies)

###### Check for impossible numeric values
###### *validate negative values in motery risk and trust or velocity fields
###### *validate the user age in days is not negative
###### Verify currency-related logic
###### *ensure amount_ src and amount_usd are positive
###### *Check that drived exchange rates fall within reasonable range
###### Validate time stamp integrity
###### *confirm no transaction time stamps occcur in the future
###### Review loaction consistency
###### *Inspect location mismatch counts to ensure the feature was generated correctly
###### *Validate that ip_country and home_country contains plausible country codes
###### Check categorical column consistency
###### *Review values to ensure no malformed or unexpected enteries
###### Validate risk scor ranges
###### Confirm fraud label integrity
###### Confirm velocity featurs do not contain negative values
###### *These need to done to ensure the data set is consistent before moving unro modelling or feature engineering

In [65]:
df.describe(include= 'all')

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
count,10836,10836,10836,10836,10836,10836,10836,10836.000000,10836.000000,10836.000000,...,10836.000000,10836,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000,10836.000000
unique,10836,1314,NaN,7,3,9,12,NaN,NaN,NaN,...,NaN,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,NaN,US,USD,NGN,mobile,NaN,NaN,NaN,...,NaN,standard,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,1433,NaN,7533,7619,1400,6016,NaN,NaN,NaN,...,NaN,7733,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,2024-05-03 11:39:15.294875+00:00,NaN,NaN,NaN,NaN,437.384918,448.258985,97.071335,...,0.398331,NaN,392.088778,0.654070,0.050941,0.268593,0.475914,0.749169,0.045484,0.090993
min,NaN,NaN,2022-10-03 18:40:59.468549+00:00,NaN,NaN,NaN,NaN,-9997.160000,7.230000,-1.000000,...,0.004000,NaN,1.000000,-0.100000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
25%,NaN,NaN,2023-08-15 06:58:52.468549+00:00,NaN,NaN,NaN,NaN,90.910000,92.600000,2.390000,...,0.209000,NaN,147.000000,0.515000,0.000000,0.169000,0.000000,0.000000,0.000000,0.000000
50%,NaN,NaN,2024-05-09 14:30:08.521080+00:00,NaN,NaN,NaN,NaN,159.080000,163.590000,3.510000,...,0.326000,NaN,272.000000,0.658000,0.000000,0.223000,0.000000,0.000000,0.000000,0.000000
75%,NaN,NaN,2025-01-29 08:52:54.047345+00:00,NaN,NaN,NaN,NaN,295.485000,302.682500,5.560000,...,0.489250,NaN,661.000000,0.894000,0.000000,0.391000,0.000000,0.000000,0.050000,0.000000
max,NaN,NaN,2025-12-16 00:13:41.468549+00:00,NaN,NaN,NaN,NaN,11942.890000,12497.900000,9999.990000,...,1.200000,NaN,1095.000000,0.999000,2.000000,0.900000,8.000000,11.000000,0.250000,1.000000


In [ ]:
# Count negative values in key numeric columns
# This helps identify potential data quality issues, as negative values in these fields may be invalid or require special handling.
neg_counts = {
    'amount_src': (df['amount_src'] < 0).sum(),
    'amount_usd': (df['amount_usd'] < 0).sum(),
    'fee': (df['fee'] < 0).sum(),
    'device_trust_score': (df['device_trust_score'] < 0).sum(),
    'txn_velocity_1h': (df['txn_velocity_1h'] < 0).sum(),
    'txn_velocity_24h': (df['txn_velocity_24h'] < 0).sum(),
    'risk_score_internal': (df['risk_score_internal'] < 0).sum()
}

neg_counts

{'amount_src': np.int64(97),
 'amount_usd': np.int64(0),
 'fee': np.int64(89),
 'device_trust_score': np.int64(186),
 'txn_velocity_1h': np.int64(186),
 'txn_velocity_24h': np.int64(0),
 'risk_score_internal': np.int64(0)}

In [67]:
# keeping value  greater than or equals to zero
df = df[
    (df['amount_src'] >= 0) &
    (df['amount_usd'] >= 0) &
    (df['fee'] >= 0) &
    (df['device_trust_score'] >= 0) &
    (df['txn_velocity_1h'] >= 0) &
    (df['txn_velocity_24h'] >= 0) &
    (df['risk_score_internal'] >= 0)
]

In [68]:
# checck correlation or exchage rates
(df[['amount_src', 'amount_usd']].corr())

,amount_src,amount_usd
amount_src,1.000000,0.989974
amount_usd,0.989974,1.000000


In [69]:
# checck correlation or exchage rates using mean ( notuce mean closseness to 1)
(df['amount_usd'] / df['amount_src']).describe()


count    10650.000000
mean         1.018225
std          0.136860
min          0.739788
25%          1.000000
50%          1.000000
75%          1.000000
max          1.250405
dtype: float64

In [70]:
# date conversion, check for unrealistic dates e,g future dates)
df['timestamp'] = df['timestamp'].dt.tz_localize(None)

In [71]:
# distribution of location mismatch
df ['location_mismatch'].value_counts()

location_mismatch
False    8884
True     1766
Name: count, dtype: int64

In [72]:
df['channel'].unique()
df['source_currency'].unique()
df['dest_currency'].unique()
df['kyc_tier'].unique()


<StringArray>
[   'standard',    'enhanced',         'low', ' standard  ',     'standrd',
 ' enhanced  ',    'STANDARD',     'unknown',     'enhancd',      ' low  ',
    'ENHANCED',         'LOW']
Length: 12, dtype: str

In [73]:
df['source_currency'].unique()

<StringArray>
['USD', 'CAD', 'GBP']
Length: 3, dtype: str

In [74]:
df['dest_currency'].unique()

<StringArray>
['CAD', 'MXN', 'CNY', 'EUR', 'INR', 'GBP', 'PHP', 'NGN', 'USD']
Length: 9, dtype: str

In [75]:
df['channel'].unique()

<StringArray>
[      'ATM',       'web',    'mobile',       'WEB',    ' web  ',    'MOBILE',
   'unknown',   'mobille', ' mobile  ',      'weeb',       'ATm',    ' ATM  ']
Length: 12, dtype: str

In [76]:
df['channel'] =df['channel'].str.lower().str.strip()

In [77]:
df['channel'].unique()

<StringArray>
['atm', 'web', 'mobile', 'unknown', 'mobille', 'weeb']
Length: 6, dtype: str

In [78]:
df['channel'] =df['channel'].replace({'web': 'web', 'weeb':'web',
                                      'mobile': 'mobile', 'mobille': 'mobile', 'atm' : 'atm'})

In [79]:
df['channel'].unique()

<StringArray>
['atm', 'web', 'mobile', 'unknown']
Length: 4, dtype: str

In [80]:
df['kyc_tier'] =df['kyc_tier'].str.lower().str.strip()

In [81]:
df['kyc_tier'].unique()

<StringArray>
['standard', 'enhanced', 'low', 'standrd', 'unknown', 'enhancd']
Length: 6, dtype: str

In [82]:
df['kyc_tier'] =df['kyc_tier'].replace({'standard': 'standard', 'standrd':'standard',
                                      'enhanced': 'enhanced', 'enhancd': 'enhanced', 'low' : 'low'})

In [83]:
df['kyc_tier'] =df['kyc_tier'].replace({'unknown': np.nan})

In [84]:
df.isna().sum()

transaction_id                0
customer_id                   0
timestamp                     0
home_country                  0
source_currency               0
dest_currency                 0
channel                       0
amount_src                    0
amount_usd                    0
fee                           0
exchange_rate_src_to_dest     0
device_id                     0
new_device                    0
ip_address                    0
ip_country                    0
location_mismatch             0
ip_risk_score                 0
kyc_tier                     28
account_age_days              0
device_trust_score            0
chargeback_history_count      0
risk_score_internal           0
txn_velocity_1h               0
txn_velocity_24h              0
corridor_risk                 0
is_fraud                      0
dtype: int64

In [85]:
df.info()

<class 'pandas.DataFrame'>
Index: 10650 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   transaction_id             10650 non-null  str           
 1   customer_id                10650 non-null  str           
 2   timestamp                  10650 non-null  datetime64[us]
 3   home_country               10650 non-null  str           
 4   source_currency            10650 non-null  str           
 5   dest_currency              10650 non-null  str           
 6   channel                    10650 non-null  str           
 7   amount_src                 10650 non-null  float64       
 8   amount_usd                 10650 non-null  float64       
 9   fee                        10650 non-null  float64       
 10  exchange_rate_src_to_dest  10650 non-null  float64       
 11  device_id                  10650 non-null  str           
 12  new_device          

In [86]:
df.to_csv(r'C:\Users\Opeyemi\OneDrive - UNC Kenan-Flagler Business School\Desktop\Data Science\Fraudulent-Transaction-Detection-for-Digital-Money-Transfer/data/cleaned_transactions.csv', index=False)


In [87]:
df = pd.read_csv(r'C:\Users\Opeyemi\OneDrive - UNC Kenan-Flagler Business School\Desktop\Data Science\Fraudulent-Transaction-Detection-for-Digital-Money-Transfer/data/cleaned_transactions.csv')


In [ ]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10650 entries, 0 to 10649
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             10650 non-null  str    
 1   customer_id                10650 non-null  str    
 2   timestamp                  10650 non-null  str    
 3   home_country               10650 non-null  str    
 4   source_currency            10650 non-null  str    
 5   dest_currency              10650 non-null  str    
 6   channel                    10650 non-null  str    
 7   amount_src                 10650 non-null  float64
 8   amount_usd                 10650 non-null  float64
 9   fee                        10650 non-null  float64
 10  exchange_rate_src_to_dest  10650 non-null  float64
 11  device_id                  10650 non-null  str    
 12  new_device                 10650 non-null  bool   
 13  ip_address                 10650 non-null  str    
 14  i

In [90]:
df.dropna(inplace=True)

In [91]:
df.info()

<class 'pandas.DataFrame'>
Index: 10622 entries, 0 to 10649
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             10622 non-null  str    
 1   customer_id                10622 non-null  str    
 2   timestamp                  10622 non-null  str    
 3   home_country               10622 non-null  str    
 4   source_currency            10622 non-null  str    
 5   dest_currency              10622 non-null  str    
 6   channel                    10622 non-null  str    
 7   amount_src                 10622 non-null  float64
 8   amount_usd                 10622 non-null  float64
 9   fee                        10622 non-null  float64
 10  exchange_rate_src_to_dest  10622 non-null  float64
 11  device_id                  10622 non-null  str    
 12  new_device                 10622 non-null  bool   
 13  ip_address                 10622 non-null  str    
 14  ip_cou